# Local Projections for Dynamic Causal Responses

**Econometrics Notebook Library · v0.1.0**

## Intuition

Instead of estimating one dynamic system and recursively iterating it forward, local projections estimate a separate regression for each horizon. The coefficient on today's shock in the $h$-step-ahead regression directly estimates the response at horizon $h$.

Reference: [Jordà, American Economic Review (2005)](https://doi.org/10.1257/0002828053828518).

## Horizon-by-horizon regression

A standard LP is

$$
Y_{t+h}=\alpha_h+\beta_h Shock_t+\Gamma_h'W_t+u_{t+h}^{(h)},
\qquad h=0,1,\ldots,H.
$$

The impulse response is the sequence $\{\beta_h\}$. Because adjacent $Y_{t+h}$ outcomes overlap as $h$ grows, $u_{t+h}^{(h)}$ is serially correlated even if the one-step innovation is not. Horizon-aware HAC or other appropriate inference is therefore essential.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.5)
pd.set_option("display.max_columns", 30)

In [ ]:
from econnotes.core import simulate_local_projection_series, local_projections

df = simulate_local_projection_series(T=700, rho=.72, theta=.8, seed=505)
lp = local_projections(df, horizons=range(0, 13), lags=2)
truth = pd.DataFrame({"horizon":np.arange(13), "truth":.8*(.72**np.arange(13))})
plot = lp.merge(truth, on="horizon")
plot.head()

In [ ]:
fig, ax = plt.subplots()
ax.plot(plot.horizon, plot.truth, marker="o", label="True AR(1) IRF")
ax.errorbar(plot.horizon, plot.estimate, yerr=1.96*plot.se, marker="o", capsize=3, label="Local projection")
ax.axhline(0, linewidth=1)
ax.set(xlabel="Horizon", ylabel="Response", title="LP estimates each horizon directly")
ax.legend();

## Specification is horizon-specific

LP robustness to dynamic misspecification is not immunity to bad identification. If the shock is not exogenous conditional on controls, every $\beta_h$ can be biased. In macro applications, shocks often come from external instruments, narrative identification, high-frequency surprises, or other designs layered on top of the LP.

## Common failure

Using plain OLS standard errors at long horizons ignores serial correlation induced by overlapping dependent variables. Another frequent error is switching between level responses and cumulative responses without changing the interpretation of $\beta_h$.

## Researcher failure checklist

- State the shock-identification argument separately from the LP estimation equation.
- Use horizon-appropriate HAC/cluster inference.
- Pre-specify lag controls or show sensitivity.
- Distinguish point responses from cumulative responses.
- Report joint confidence bands when the research question concerns the full response path rather than isolated horizons.